# Sesión 4 · Datasets y fine-tuning

Carga y prepara datos, y afina un modelo preentrenado con `Trainer`.

Objetivos:
1. Cargar y transformar un dataset.
2. Afinar un modelo de clasificación.
3. Evaluar con accuracy/F1.

In [ ]:
%pip install -q transformers datasets evaluate accelerate

## 1 · Cargar y explorar

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")
print(dataset)
print(dataset["train"][0])

## 2 · Subconjunto y tokenización

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train = dataset["train"].shuffle(seed=42).select(range(1000))
test = dataset["test"].shuffle(seed=42).select(range(200))

def tokenizar(e):
    return tokenizer(e["text"], truncation=True, padding="max_length")

train = train.map(tokenizar, batched=True)
test = test.map(tokenizar, batched=True)
print(len(train), len(test))

## 3 · Fine-tuning con Trainer

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate, numpy as np

modelo = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

args = TrainingArguments(
    output_dir="./resultados", evaluation_strategy="epoch",
    num_train_epochs=1, per_device_train_batch_size=16,
)

accuracy = evaluate.load("accuracy")

def metricas(eval_pred):
    logits, etiquetas = eval_pred
    return accuracy.compute(predictions=np.argmax(logits, -1), references=etiquetas)

trainer = Trainer(model=modelo, args=args,
                  train_dataset=train, eval_dataset=test,
                  compute_metrics=metricas)
trainer.train()
print(trainer.evaluate())

## 4 · Guardar y reutilizar

In [ ]:
trainer.save_model("./mi_modelo")

from transformers import pipeline
clasificador = pipeline("sentiment-analysis", model="./mi_modelo")
print(clasificador("Me ha encantado el producto."))

## Práctica

Ajusta un modelo **en español** (p. ej. `dccuchile/bert-base-spanish-wwm-uncased`) sobre reseñas en español y compáralo con el zero-shot de la sesión 1.

## Reto

Afina un modelo de **NER** (`AutoModelForTokenClassification`) o de **clasificación multietiqueta**.

In [ ]:
# TODO: fine-tuning en español y comparación con zero-shot